# JCI Aso - Plan of Action Survey 2027
## Data Grain Reorganization & Cleaning Pipeline

### Educational Notebook: Transforming Survey Grain from Aggregated to Atomic

**Context & Objective:**
In survey data analysis, a common design challenge with raw exports (from tools like Google Forms) is that **multi-select checkbox questions** pack multiple choices into a single cell, separated by commas (e.g., `"Individual Development, Business and Entrepreneurship, International Opportunity"`).

This causes significant analytical issues:
1. **Pivot Tables & Grouping:** Excel and BI tools cannot aggregate, filter, or count individual choices accurately without messy string formulas.
2. **Cross-Tabulation:** You cannot easily cross-tabulate multi-select preferences against member demographics (e.g., Active vs. Alumni).
3. **Inconsistent Grain:** Demographic attributes exist at the *respondent grain* (1 row per member), while checkbox answers exist at the *choice grain* (multiple choices per member).

**Our Target Architecture:**
- Reorganize the dataset so that each row represents **one choice per multi-select question**.
- Assign a clean, standardized identifier: `Response_ID` (`R001` to `R022`).
- Add a `Choice_Row_Number` (1, 2, 3...) to identify the row sequence for each member.
- **Drop `Timestamp` and `Email`** to anonymize responses and focus strictly on analytical variables.
- Replicate member demographics (`Member_Category`, `Tenure`, `Activity_Level`) across all rows of that respondent so dimensional slicing works.
- Place scalar metrics (1-5 Star Ratings and qualitative text comments) on **Row 1** per respondent (`Choice_Row_Number == 1`) with subsequent rows blank (`None`), preventing accidental metric inflation (double-counting averages) in Excel.
- Standardize text, typos (`'Submmit'` -> `'Summit'`), casing, and export clean multi-sheet Excel and CSV files.

### Step 1: Environment Setup & Library Imports
We use `pandas` for tabular modeling and `openpyxl` for high-fidelity Excel manipulation.

In [ ]:
import os
import sys
import openpyxl
import pandas as pd

# Print library versions
print(f"Pandas version: {pd.__version__}")
print(f"Openpyxl version: {openpyxl.__version__}")

# Configure file paths
INPUT_FILE = "Plan_of_Action.xlsx"
OUTPUT_EXCEL = "Plan_of_Action_Cleaned_Grain.xlsx"
OUTPUT_CSV = "Plan_of_Action_Cleaned_Grain.csv"

print(f"Input raw file: {INPUT_FILE}")
print(f"Target output Excel: {OUTPUT_EXCEL}")

### Step 2: Loading & Inspecting the Raw Survey Data
Let's load the workbook and inspect the sheet dimensions and column headers.

In [ ]:
# Load raw workbook using openpyxl
wb = openpyxl.load_workbook(INPUT_FILE)
sheet = wb.active
raw_rows = list(sheet.iter_rows(values_only=True))

raw_headers = raw_rows[0]
raw_data = raw_rows[1:]

print(f"Active Sheet: {sheet.title}")
print(f"Total columns: {len(raw_headers)}")
print(f"Total respondent submissions: {len(raw_data)}\n")

# Display raw columns
for i, col in enumerate(raw_headers, 1):
    print(f"Col {i:02d}: {col}")

### Step 3: Question Classification & Schema Architecture

The 24 columns in the raw sheet map directly to the 23 survey questions:
- **Dropped Fields (Cols 1 & 2):** `Timestamp` and `Email` are dropped during processing for member privacy and clean modeling.
- **Demographics (Cols 3-5):** Category (Active/Alumni), Tenure, Activity Level.
- **Multi-Select Checkboxes (Cols 6, 7, 17, 19, 21, 22):** Opportunities, Past Projects, Benefits Gained, Programs to Discontinue, 2027 Innovations, 2027 Proposed Programs.
- **Scalar Ratings (Cols 9-16):** 1-5 star ratings for 8 distinct projects.
- **Qualitative Comments (Cols 8, 18, 20, 23, 24):** Satisfaction text, project reasons, discontinuation reasons, improvement suggestions, general comments.

In [ ]:
# Dictionary of standardized options and known typo corrections
STANDARDIZED_MAPPINGS = {
    'opportunity': {
        'individual development': 'Individual Development',
        'business and entrepreneurship': 'Business and Entrepreneurship',
        'international opportunity': 'International Opportunity',
        'community development': 'Community Development'
    },
    'projects': {
        'save a soul': 'Save a Soul',
        'baba and yara': 'Baba and Yara',
        'membership development submmit (mds)': 'Membership Development Summit (MDS)',
        'membership development summit (mds)': 'Membership Development Summit (MDS)',
        'secondary school debate': 'Secondary School Debate',
        'educate a child': 'Educate a Child',
        'quality leadership value (qlv)': 'Quality Leadership Value (QLV)',
        'international women day': "International Women's Day",
        'international women day ': "International Women's Day",
        'international women\'s day': "International Women's Day",
        'world down syndrome day': 'World Down Syndrome Day',
        'world down syndrome day ': 'World Down Syndrome Day'
    },
    'benefits': {
        'networking opportunities': 'Networking opportunities',
        'community engagement': 'Community engagement',
        'fulfilment for serving a cause': 'Fulfilment for serving a cause',
        'new skills': 'New skills',
        'personal growth': 'Personal growth'
    },
    'innovations': {
        'international collaborations': 'International collaborations',
        'strategic partnerships with other organizations': 'Strategic partnerships with other organizations',
        'research and apply for grants and funding opportunities': 'Research and apply for grants and funding opportunities',
        'technology-enabled projects': 'Technology-enabled projects',
        'focus on sustainable development': 'Focus on sustainable development'
    },
    'new_programs': {
        'international exchange programme': 'International exchange programme',
        'economic development': 'Economic development',
        'healthcare and wellness': 'Healthcare and wellness',
        'digital skills': 'Digital Skills',
        'cultural exchange': 'Cultural exchange',
        'business clinic': 'Business Clinic'
    }
}
print("Standardization dictionary configured successfully.")

### Step 4: Text Cleaning & Multi-Choice Parsing Helper Functions
We define two cleaning functions:
1. `clean_scalar_text(val)`: Handles nulls, strips whitespace, converts 'None'/'Nil'/'NA' to `None`, and normalizes curly quotes.
2. `clean_multi_choice(val, mapping_key)`: Splits comma-separated strings, strips whitespace around choices, filters out blanks, and applies typo corrections.

In [ ]:
def clean_scalar_text(val):
    """Clean and normalize single-valued free-text and categorical fields."""
    if val is None:
        return None
    s = str(val).strip()
    if not s or s.lower() in ['none', 'blank', 'nil', 'n/a', 'na']:
        return None
    # Replace smart/curly quotes with standard ASCII
    s = (s.replace('’', "'")
          .replace('‘', "'")
          .replace('“', '"')
          .replace('”', '"')
          .replace('–', '-')
          .replace('—', '-'))
    return s

def clean_multi_choice(val, mapping_key):
    """Parse and standardize comma-separated multi-select choices into a list."""
    if val is None:
        return []
    s = str(val).strip()
    if not s or s.lower() in ['none', 'blank', 'nil', 'n/a', 'na']:
        return []
    
    parts = [p.strip() for p in s.split(',') if p.strip()]
    cleaned = []
    mapping = STANDARDIZED_MAPPINGS.get(mapping_key, {})
    
    for p in parts:
        lower_p = p.lower()
        if lower_p in ['none', 'blank', 'nil', 'n/a', 'na']:
            continue
        if lower_p in mapping:
            cleaned.append(mapping[lower_p])
        else:
            norm_p = (p.replace('’', "'")
                       .replace('‘', "'")
                       .replace('“', '"')
                       .replace('”', '"'))
            cleaned.append(norm_p)
    return cleaned

# Quick demonstration on raw string
sample_raw = "save a soul, Membership Development Submmit (MDS), International Women Day"
print("Raw sample:", sample_raw)
print("Cleaned result:", clean_multi_choice(sample_raw, 'projects'))

### Step 5: Grain Transformation & Parallel Row Expansion

**How the Algorithm Works:**
1. For each respondent, parse all 6 multi-select checkbox columns into Python lists.
2. Determine `max_choices = max(1, len(col1), len(col2), ...)`.
3. **Drop `Timestamp` and `Email`** (omitted from output row dictionaries).
4. Loop `i` from `0` to `max_choices - 1`:
   - Multi-select columns: If `i < len(choices)`, place `choices[i]`, otherwise place `None` (blank).
   - Demographics: Repeated across all rows for that respondent so demographic grouping works.
   - Scalar ratings and text: Placed on **Row 1 only** (`i == 0`), leaving subsequent rows blank (`None`) to prevent double-counting averages.
5. In parallel, build a **Fully Denormalized** version where ratings are filled down (useful if connecting directly to BI tools like Tableau or Power BI).

In [ ]:
modeled_rows = []
denormalized_rows = []

for r_idx, r in enumerate(raw_data, 1):
    resp_id = f"R{r_idx:03d}"
    
    # Demographics (Timestamp & Email dropped)
    category = clean_scalar_text(r[2])
    tenure = clean_scalar_text(r[3])
    activity = clean_scalar_text(r[4])
    
    # Multi-Select Questions
    opp_areas = clean_multi_choice(r[5], 'opportunity')
    proj_inv = clean_multi_choice(r[6], 'projects')
    benefits = clean_multi_choice(r[16], 'benefits')
    disc_projs = clean_multi_choice(r[18], 'projects')
    new_ideas = clean_multi_choice(r[20], 'innovations')
    new_projs = clean_multi_choice(r[21], 'new_programs')
    
    # Ratings (Cols 8-15 / Questions 8-15)
    rate_sas = r[8]
    rate_by = r[9]
    rate_mds = r[10]
    rate_deb = r[11]
    rate_eac = r[12]
    rate_qlv = r[13]
    rate_iwd = r[14]
    rate_wdsd = r[15]
    
    # Qualitative Comments
    sat_text = clean_scalar_text(r[7])
    proj_reason_text = clean_scalar_text(r[17])
    disc_reason_text = clean_scalar_text(r[19])
    improve_areas = clean_scalar_text(r[22])
    other_comments = clean_scalar_text(r[23])
    
    # Calculate expansion depth
    max_choices = max(
        1,
        len(opp_areas),
        len(proj_inv),
        len(benefits),
        len(disc_projs),
        len(new_ideas),
        len(new_projs)
    )
    
    for i in range(max_choices):
        cur_opp = opp_areas[i] if i < len(opp_areas) else None
        cur_proj = proj_inv[i] if i < len(proj_inv) else None
        cur_benefit = benefits[i] if i < len(benefits) else None
        cur_disc = disc_projs[i] if i < len(disc_projs) else None
        cur_idea = new_ideas[i] if i < len(new_ideas) else None
        cur_new_proj = new_projs[i] if i < len(new_projs) else None
        
        # Primary Modeled Grain: Timestamp & Email dropped, ratings on Row 1 only
        modeled_row = {
            "Response_ID": resp_id,
            "Choice_Row_Number": i + 1,
            "Member_Category": category,
            "Tenure": tenure,
            "Activity_Level": activity,
            "Opportunity_Areas": cur_opp,
            "Projects_Involved": cur_proj,
            "Satisfaction_Text": sat_text if i == 0 else None,
            "Rate_SaveASoul": rate_sas if i == 0 else None,
            "Rate_BabaYara": rate_by if i == 0 else None,
            "Rate_MDS": rate_mds if i == 0 else None,
            "Rate_Debate": rate_deb if i == 0 else None,
            "Rate_EducateChild": rate_eac if i == 0 else None,
            "Rate_QualityLeadershipValue": rate_qlv if i == 0 else None,
            "Rate_InternationalWomenDay": rate_iwd if i == 0 else None,
            "Rate_WorldDownSyndromeDay": rate_wdsd if i == 0 else None,
            "Benefits_Gained": cur_benefit,
            "Project_and_Reason_Text": proj_reason_text if i == 0 else None,
            "Programs_To_Discontinue": cur_disc,
            "Reason_For_Discontinuation": disc_reason_text if i == 0 else None,
            "New_Ideas": cur_idea,
            "New_Projects_Suggested": cur_new_proj,
            "Improve_Areas": improve_areas if i == 0 else None,
            "Other_Comments": other_comments if i == 0 else None,
        }
        modeled_rows.append(modeled_row)
        
        # Fully Denormalized: All ratings and comments filled down across all rows
        denorm_row = {
            "Response_ID": resp_id,
            "Choice_Row_Number": i + 1,
            "Member_Category": category,
            "Tenure": tenure,
            "Activity_Level": activity,
            "Opportunity_Areas": cur_opp,
            "Projects_Involved": cur_proj,
            "Satisfaction_Text": sat_text,
            "Rate_SaveASoul": rate_sas,
            "Rate_BabaYara": rate_by,
            "Rate_MDS": rate_mds,
            "Rate_Debate": rate_deb,
            "Rate_EducateChild": rate_eac,
            "Rate_QualityLeadershipValue": rate_qlv,
            "Rate_InternationalWomenDay": rate_iwd,
            "Rate_WorldDownSyndromeDay": rate_wdsd,
            "Benefits_Gained": cur_benefit,
            "Project_and_Reason_Text": proj_reason_text,
            "Programs_To_Discontinue": cur_disc,
            "Reason_For_Discontinuation": disc_reason_text,
            "New_Ideas": cur_idea,
            "New_Projects_Suggested": cur_new_proj,
            "Improve_Areas": improve_areas,
            "Other_Comments": other_comments,
        }
        denormalized_rows.append(denorm_row)

df_modeled = pd.DataFrame(modeled_rows)
df_denorm = pd.DataFrame(denormalized_rows)

print(f"Transformation Complete!")
print(f"Original submissions: {len(raw_data)} rows")
print(f"New Modeled Grain: {len(df_modeled)} rows x {df_modeled.shape[1]} columns (Timestamp and Email dropped)")

### Alternative Method: Dropping Columns using Pandas `df.drop()`
For your learning, if you had already loaded a DataFrame containing `Timestamp` and `Email`, you could also drop them using pandas directly:
```python
columns_to_drop = ['Timestamp', 'Email']
df_modeled.drop(columns=columns_to_drop, inplace=True, errors='ignore')
```
In our pipeline above, we excluded them during row dictionary construction for maximum memory efficiency.

### Step 6: Quality Assurance & Verification
Let's inspect sample rows for Respondent `R001` to verify that:
1. `Timestamp` and `Email` are completely absent.
2. Multi-select choices are placed 1-choice-per-row.
3. Shorter lists leave trailing choice rows blank (`NaN`/`None`).
4. Ratings and single text comments appear on Row 1 only.
5. Frequency counts across choices match raw counts with zero data loss.

In [ ]:
# Preview R001 choice expansion (Note: Timestamp and Email are gone)
cols_to_preview = [
    'Response_ID', 'Choice_Row_Number', 'Member_Category', 
    'Opportunity_Areas', 'Projects_Involved', 'Benefits_Gained', 
    'New_Ideas', 'New_Projects_Suggested', 'Rate_BabaYara'
]
df_modeled[df_modeled['Response_ID'] == 'R001'][cols_to_preview]

In [ ]:
# Frequency counts for multi-select columns
print("=== Frequency: Opportunity Areas ===")
print(df_modeled['Opportunity_Areas'].value_counts(dropna=True))

print("\n=== Frequency: Projects Involved ===")
print(df_modeled['Projects_Involved'].value_counts(dropna=True))

print("\n=== Frequency: 2027 Proposed Programs ===")
print(df_modeled['New_Projects_Suggested'].value_counts(dropna=True))

# Check average ratings (computed on Row 1 only to ensure no double-counting)
print("\n=== Average Project Ratings (Out of 5 Stars) ===")
rating_cols = [c for c in df_modeled.columns if c.startswith('Rate_')]
row1_subset = df_modeled[df_modeled['Choice_Row_Number'] == 1]
for rc in rating_cols:
    avg = row1_subset[rc].mean()
    print(f"{rc.replace('Rate_', '')}: {avg:.2f} ★")

### Step 7: Creating the Data Dictionary & Metadata Sheet
A clean data asset includes a data dictionary describing each column, question number, data type, and valid options.

In [ ]:
dictionary_data = [
    {"Column_Name": "Response_ID", "Question_Num": "-", "Type": "Identifier", "Description": "Unique identifier for each respondent (R001 to R022)", "Canonical_Options": "R001 - R022"},
    {"Column_Name": "Choice_Row_Number", "Question_Num": "-", "Type": "Index", "Description": "Ordinal position of the choice row for this respondent (1, 2, ...)", "Canonical_Options": "1 to 8"},
    {"Column_Name": "Member_Category", "Question_Num": "Q2", "Type": "Multiple Choice", "Description": "Category of JCI Aso membership", "Canonical_Options": "Active member (18 - 40 years), Alumni (Above 40)"},
    {"Column_Name": "Tenure", "Question_Num": "Q3", "Type": "Multiple Choice", "Description": "Duration of membership in JCI Aso", "Canonical_Options": "1 - 3 years, 3 - 5 years, 5 years and above"},
    {"Column_Name": "Activity_Level", "Question_Num": "Q4", "Type": "Multiple Choice", "Description": "Self-reported involvement level in JCI Aso activities", "Canonical_Options": "Very active, Somewhat active, Not active"},
    {"Column_Name": "Opportunity_Areas", "Question_Num": "Q5", "Type": "Checkboxes (Multi-select)", "Description": "Areas of opportunity member wants to benefit more from (1 choice per row)", "Canonical_Options": "Individual Development, Business and Entrepreneurship, International Opportunity, Community Development"},
    {"Column_Name": "Projects_Involved", "Question_Num": "Q6", "Type": "Checkboxes (Multi-select)", "Description": "JCI Aso projects/programmes member has been involved with in the past (1 choice per row)", "Canonical_Options": "Save a Soul, Baba and Yara, Membership Development Summit (MDS), Secondary School Debate, Educate a Child, Quality Leadership Value (QLV), International Women's Day, World Down Syndrome Day"},
    {"Column_Name": "Satisfaction_Text", "Question_Num": "Q7", "Type": "Short Answer", "Description": "Satisfaction sentiment with project impact and outcomes (Row 1 per respondent)", "Canonical_Options": "Free text"},
    {"Column_Name": "Rate_SaveASoul", "Question_Num": "Q8", "Type": "Rating (1-5 Stars)", "Description": "Impact rating for Save a Soul project", "Canonical_Options": "1 to 5"},
    {"Column_Name": "Rate_BabaYara", "Question_Num": "Q9", "Type": "Rating (1-5 Stars)", "Description": "Impact rating for Baba and Yara project", "Canonical_Options": "1 to 5"},
    {"Column_Name": "Rate_MDS", "Question_Num": "Q10", "Type": "Rating (1-5 Stars)", "Description": "Impact rating for Membership Development Summit (MDS)", "Canonical_Options": "1 to 5"},
    {"Column_Name": "Rate_Debate", "Question_Num": "Q11", "Type": "Rating (1-5 Stars)", "Description": "Impact rating for Secondary School Debate", "Canonical_Options": "1 to 5"},
    {"Column_Name": "Rate_EducateChild", "Question_Num": "Q12", "Type": "Rating (1-5 Stars)", "Description": "Impact rating for Educate a Child", "Canonical_Options": "1 to 5"},
    {"Column_Name": "Rate_QualityLeadershipValue", "Question_Num": "Q13", "Type": "Rating (1-5 Stars)", "Description": "Impact rating for Quality Leadership Value (QLV)", "Canonical_Options": "1 to 5"},
    {"Column_Name": "Rate_InternationalWomenDay", "Question_Num": "Q14", "Type": "Rating (1-5 Stars)", "Description": "Impact rating for International Women's Day", "Canonical_Options": "1 to 5"},
    {"Column_Name": "Rate_WorldDownSyndromeDay", "Question_Num": "Q15", "Type": "Rating (1-5 Stars)", "Description": "Impact rating for World Down Syndrome Day", "Canonical_Options": "1 to 5"},
    {"Column_Name": "Benefits_Gained", "Question_Num": "Q16", "Type": "Checkboxes (Multi-select)", "Description": "Benefits gained participating in projects/programmes (1 choice per row)", "Canonical_Options": "Networking opportunities, Community engagement, Fulfilment for serving a cause, New skills, Personal growth"},
    {"Column_Name": "Project_and_Reason_Text", "Question_Num": "Q17", "Type": "Short Answer", "Description": "Specific projects and reasons for member benefits", "Canonical_Options": "Free text"},
    {"Column_Name": "Programs_To_Discontinue", "Question_Num": "Q18", "Type": "Checkboxes (Multi-select)", "Description": "Projects or programmes recommended for discontinuation", "Canonical_Options": "Save a Soul, Baba and Yara, MDS, Secondary School Debate, Educate a Child, QLV, International Women's Day, World Down Syndrome Day"},
    {"Column_Name": "Reason_For_Discontinuation", "Question_Num": "Q19", "Type": "Short Answer", "Description": "Reasons for proposing project discontinuation", "Canonical_Options": "Free text"},
    {"Column_Name": "New_Ideas", "Question_Num": "Q20", "Type": "Checkboxes (Multi-select)", "Description": "New ideas or innovations for JCI Aso in 2027 (1 choice per row)", "Canonical_Options": "International collaborations, Strategic partnerships with other organizations, Research and apply for grants and funding opportunities, Technology-enabled projects, Focus on sustainable development"},
    {"Column_Name": "New_Projects_Suggested", "Question_Num": "Q21", "Type": "Checkboxes (Multi-select)", "Description": "New projects or programmes introduced in 2027 (1 choice per row)", "Canonical_Options": "International exchange programme, Economic development, Healthcare and wellness, Digital Skills, Cultural exchange, Business Clinic"},
    {"Column_Name": "Improve_Areas", "Question_Num": "Q22", "Type": "Paragraph", "Description": "Suggested improvement areas for JCI Aso operations", "Canonical_Options": "Free text"},
    {"Column_Name": "Other_Comments", "Question_Num": "Q23", "Type": "Short Answer", "Description": "Final comments or suggestions for 2027 Plan of Action", "Canonical_Options": "Free text"}
]
df_dict = pd.DataFrame(dictionary_data)
print(f"Data dictionary created with {len(df_dict)} documented fields.")
df_dict.head(10)

### Step 8: Saving the Cleaned Dataset in the Current Directory
We now write the multi-sheet Excel file (`Plan_of_Action_Cleaned_Grain.xlsx`) and CSV file (`Plan_of_Action_Cleaned_Grain.csv`).
- **Sheet 1: `Modeled_Grain`**: The primary atomic dataset (1 choice per row, ratings on row 1 to prevent double-counting).
- **Sheet 2: `Fully_Denormalized`**: The fully filled-down dataset (for direct BI connections).
- **Sheet 3: `Data_Dictionary`**: The metadata and codebook reference.

In [ ]:
# Final Code Cell: Exporting cleaned data to the current directory
print(f"Writing multi-sheet workbook to: {OUTPUT_EXCEL} ...")
with pd.ExcelWriter(OUTPUT_EXCEL, engine='openpyxl') as writer:
    df_modeled.to_excel(writer, sheet_name="Modeled_Grain", index=False)
    df_denorm.to_excel(writer, sheet_name="Fully_Denormalized", index=False)
    df_dict.to_excel(writer, sheet_name="Data_Dictionary", index=False)

# Also export standard CSV
df_modeled.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')

print(f"SUCCESS! Cleaned datasets saved:")
print(f"  1. {os.path.abspath(OUTPUT_EXCEL)}")
print(f"  2. {os.path.abspath(OUTPUT_CSV)}")